In [1]:
from pathlib import Path
import json
import shutil

LAST_NAME = "Mejos"
STUDENT_ID = "TUPM-26-3605"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

dataset_path = DATA_DIR / "weblog.csv"
filtered_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_filtered.csv"
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_summary.json"

records_to_read = 1000 + ID_SUM
target_class = (SEED_DIGIT % 4) + 2

processed = 0
invalid = 0
in_class = 0
status_counts = {}
url_counts = {}

with open(dataset_path, "r", encoding="utf-8") as file:
    header = file.readline()

    with open(filtered_path, "w", encoding="utf-8") as output:
        output.write(header)

        for _ in range(records_to_read):
            line = file.readline()

            if not line:
                break

            processed += 1
            fields = line.strip().split(",")

            if len(fields) == 4 and fields[3].isdigit():
                url = fields[2]
                status = fields[3]

                if int(status) // 100 == target_class:
                    in_class += 1
                    output.write(line.strip() + "\n")

                    if status in status_counts:
                        status_counts[status] += 1
                    else:
                        status_counts[status] = 1

                    if url in url_counts:
                        url_counts[url] += 1
                    else:
                        url_counts[url] = 1
            else:
                invalid += 1

# Five most requested URLs in the target class
url_list = []
for url, count in url_counts.items():
    url_list.append([count, url])
url_list.sort()
url_list.reverse()
top_urls = url_list[0:5]

# JSON summary and backup
summary = {
    "records_processed": processed,
    "invalid_records": invalid,
    "target_class": f"{target_class}xx",
    "records_in_class": in_class,
    "count_per_status": status_counts,
    "top_five_urls": top_urls
}

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

backup_path = BACKUP_DIR / json_path.name
shutil.copy2(json_path, backup_path)

print("Records processed:", processed)
print("Invalid records:", invalid)
print("Target status class:", f"{target_class}xx")
print("Records in target class:", in_class)
print("Count per status code:", status_counts)
print("Five most requested URLs:")
for count, url in top_urls:
    print(f"  {count} x {url}")
print("Filtered CSV file:", filtered_path)
print("JSON summary file:", json_path)
print("Backup verification:", backup_path, "| exists:", backup_path.exists())

print("\n=== FINAL OUTPUT ===")
print(f"Of {processed} log records, {in_class} belong to the {target_class}xx class.")
print(f"Summary saved to {json_path.name} and backed up: {backup_path.exists()}")

Records processed: 1022
Invalid records: 0
Target status class: 3xx
Records in target class: 165
Count per status code: {'302': 144, '304': 21}
Five most requested URLs:
  46 x GET / HTTP/1.1
  35 x POST /process.php HTTP/1.1
  34 x GET /contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest HTTP/1.1
  9 x GET /home.php HTTP/1.1
  7 x POST /action.php HTTP/1.1
Filtered CSV file: output\Mejos_TUPM-26-3605_weblog_filtered.csv
JSON summary file: output\Mejos_TUPM-26-3605_weblog_summary.json
Backup verification: backup\Mejos_TUPM-26-3605_weblog_summary.json | exists: True

=== FINAL OUTPUT ===
Of 1022 log records, 165 belong to the 3xx class.
Summary saved to Mejos_TUPM-26-3605_weblog_summary.json and backed up: True
